# Audyt modeli: wspólne dane, cechy i walidacja

Zastępuje historyczny eksperyment 03; poprzedni kod jest zachowany w historii Git.
Cel: online nowcasting y(t), proces x(t) dostępny w chwili t, historia pyłu najwyżej
z t−10 s. Nie jest to prognoza wielokrokowa. Dostępność sygnałów w instalacji wymaga potwierdzenia.

Trzy modele: persistence, procesowy XGBoost, procesowy XGBoost z historią pyłu.
Dwa rozwijane podziały walidacji i jeden wspólny test, granice ustalone przed FE.
Bez strojenia na teście, bez imputacji i bez przechodzenia cech przez luki.
Test historyczny był już oglądany: wyniki eksploracyjne, nie niezależna walidacja końcowa.

Kod: `src/model_features.py`, `scripts/run_model_audit.py`; raport: `docs/model_audit.md`.


In [ ]:
from pathlib import Path
import sys
import subprocess
import json
import pandas as pd
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
AUDIT_OUTPUT = PROJECT_ROOT / 'data' / 'processed' / 'audit_v3'


Uruchomienie poniższej komórki przelicza wszystkie modele i zapisuje nowe artefakty tylko w ignorowanym audit_v3. Dane wejściowe pozostają bez zmian.


In [ ]:
subprocess.run([sys.executable, '-u', str(PROJECT_ROOT / 'scripts' / 'run_model_audit.py')], cwd=PROJECT_ROOT, check=True)


In [ ]:
results = pd.read_csv(AUDIT_OUTPUT / 'metrics.csv')
results


In [ ]:
stratified = pd.read_csv(AUDIT_OUTPUT / 'stratified_metrics.csv')
stratified


In [ ]:
report = json.loads((AUDIT_OUTPUT / 'report.json').read_text(encoding='utf-8'))
assert report['status'] == 'complete' and report['inputs_unchanged']
report['bootstrap']


Historyczny baseline z 01 zawierał przeciek i nie jest punktem odniesienia.
Wszystkie nowe metryki odnoszą się do tych samych wierszy w danym podziale.
Średnie/diff sygnałów procesowych mogą używać x(t); cechy pyłu wykluczają y(t).
Bootstrap resampluje całe obserwowane dni, lecz przy małej liczbie dni jego zakresy
są opisowe. Wnioski o optymalizacji ECO wymagają osobnej analizy przyczynowej.
